# Adaptive Exploration: UCB Algorithm

## 📚 Learning Objectives

By completing this notebook, you will:
- Use UCB and adaptive exploration
- Apply to bandits and simple MDPs

## 🔗 Prerequisites

- ✅ Basic Python
- ✅ Basic NumPy/Pandas (when applicable)

---

## Official Structure Reference

This notebook supports **Course 09, Unit 4** requirements from `DETAILED_UNIT_DESCRIPTIONS.md`.

---


# Adaptive Exploration: UCB Algorithm
## AIAT 123 - Reinforcement Learning

---

## 📚 Learning Objectives | أهداف التعلم

This notebook demonstrates key concepts through hands-on examples.

By completing this notebook, you will:
- Understand Upper Confidence Bound (UCB) algorithm
- Implement UCB for multi-armed bandit problem
- Compare UCB with epsilon-greedy
- Apply UCB to real-world scenarios

---

## 🔗 Prerequisites | المتطلبات الأساسية

- ✅ Python 3.8+ installed
- ✅ Required libraries (see `requirements.txt`)
- ✅ Basic Python knowledge

---

## Real-World Context

You are optimizing an online advertising campaign using UCB to balance exploring new ad variants and exploiting known high-performing ads.

---


## UCB Algorithm

Implement Upper Confidence Bound algorithm.


## 📥 Inputs & 📤 Outputs | المدخلات والمخرجات

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---


In [ ]:
import numpy as np
import matplotlib.pyplot as plt



## 🌍 Real-World Worked Example — A/B Testing with Multi-Armed Bandits

**Industry context:**
- Netflix shows different thumbnails to different users and picks the best one using Thompson Sampling — same as this example
- Google Ads uses epsilon-greedy to balance showing ads that already work vs. testing new ones
- Booking.com runs 1000+ bandit experiments simultaneously

We compare **ε-greedy**, **UCB**, and **Thompson Sampling** on a real A/B test simulation.

In [ ]:
import numpy as np, matplotlib.pyplot as plt

np.random.seed(42)
# ── Scenario: 5 website variants (like Netflix thumbnails)
# True click-through rates (unknown to agent)
TRUE_RATES = [0.15, 0.22, 0.35, 0.18, 0.28]   # Variant 3 is best (35%)
N_ARMS   = len(TRUE_RATES)
N_ROUNDS = 2000

def pull(arm): return 1 if np.random.rand() < TRUE_RATES[arm] else 0

def epsilon_greedy(eps):
    counts = np.zeros(N_ARMS); totals = np.zeros(N_ARMS); rewards = []
    for t in range(N_ROUNDS):
        arm = np.random.randint(N_ARMS) if np.random.rand()<eps else np.argmax(totals/(counts+1e-9))
        r = pull(arm); counts[arm]+=1; totals[arm]+=r; rewards.append(r)
    return rewards

def ucb():
    counts = np.zeros(N_ARMS); totals = np.zeros(N_ARMS); rewards = []
    for arm in range(N_ARMS): r=pull(arm); counts[arm]=1; totals[arm]+=r; rewards.append(r)
    for t in range(N_ARMS, N_ROUNDS):
        ucb_vals = totals/counts + np.sqrt(2*np.log(t+1)/counts)
        arm = np.argmax(ucb_vals)
        r = pull(arm); counts[arm]+=1; totals[arm]+=r; rewards.append(r)
    return rewards

def thompson():
    alpha = np.ones(N_ARMS); beta_ = np.ones(N_ARMS); rewards = []
    for _ in range(N_ROUNDS):
        samples = np.random.beta(alpha, beta_)
        arm = np.argmax(samples)
        r = pull(arm)
        if r: alpha[arm]+=1
        else: beta_[arm]+=1
        rewards.append(r)
    return rewards

strategies = {
    'ε-greedy (ε=0.1)': epsilon_greedy(0.1),
    'UCB':               ucb(),
    'Thompson Sampling': thompson(),
}

def cumavg(lst): return np.cumsum(lst) / np.arange(1,len(lst)+1)

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
for name, r in strategies.items():
    plt.plot(cumavg(r), label=name)
plt.axhline(max(TRUE_RATES), color='k', linestyle='--', label=f"Best possible ({max(TRUE_RATES)})")
plt.title("Cumulative Average Reward (A/B Test)"); plt.xlabel("Round"); plt.ylabel("CTR"); plt.legend()
plt.subplot(1,2,2)
for name, r in strategies.items():
    regret = np.cumsum([max(TRUE_RATES)-pull(np.argmax(TRUE_RATES)) for _ in r]) - np.cumsum(r) + np.cumsum([pull(np.argmax(TRUE_RATES)) for _ in r])
    regret_simple = np.cumsum(max(TRUE_RATES) - np.array(r))
    plt.plot(regret_simple, label=name)
plt.title("Cumulative Regret (lower=better)"); plt.xlabel("Round"); plt.ylabel("Regret"); plt.legend()
plt.tight_layout(); plt.show()
print(f"\nTotal rewards — ε-greedy: {sum(strategies['ε-greedy (ε=0.1)'])}, UCB: {sum(strategies['UCB'])}, Thompson: {sum(strategies['Thompson Sampling'])}")

## 📚 References & Further Reading

**Book:** Sutton & Barto — [RL Introduction Ch.2: Multi-armed Bandits](http://incompleteideas.net/book/the-book-2nd.html)
**Paper:** Auer et al. (2002) — [Finite-time Analysis of UCB](https://link.springer.com/article/10.1023/A:1013689704352)

**Real-World Usage:**
- Google uses Thompson Sampling in Google Ads for A/B testing at scale
- Netflix uses bandit algorithms to pick which thumbnail to show each user

**State-of-the-Art:** ε-greedy and UCB are still dominant in industry A/B testing systems.

## 📝 Summary

You compared **exploration strategies**: ε-greedy (simple, effective), UCB (optimism under uncertainty), and Thompson Sampling (Bayesian). The exploration-exploitation tradeoff is central to RL. Real applications: A/B testing at Netflix, ad bidding at Google, recommendation engines at Spotify.